# 00 — Setup and data pipeline

**Project:** Quantitatively Audited Explainable AI for Brain Tumor MRI Classification
(with a multi-task segmentation extension)

This notebook handles everything that has to exist before any model is trained:

1. install the few libraries Kaggle doesn't ship, and confirm the GPU is visible
2. download the Figshare brain tumor dataset (Cheng et al., CC BY 4.0) into `/kaggle/working/`
3. parse all 3064 `.mat` files into a single `manifest.csv` with a **patient-level** train/val/test split
4. check the split for patient leakage and report class balance
5. look at real images with their tumor masks overlaid, to confirm the masks line up

It trains nothing. It exists so that every later notebook (baseline, explanations, audit,
uncertainty, multi-task, comparative audit) reads **the same manifest, the same split and the
same config**. Notebook 7 compares two models' explanation metrics, and that comparison only
means something if both models were trained and evaluated on identical patients.

**What this notebook writes to `/kaggle/working/`:**

```
/kaggle/working/
├── configs/baseline_resnet18.yaml   # hyperparameters shared by all notebooks
├── src/data/parse_mat.py            # .mat -> TumorSample reader
├── src/utils/seed.py                # set_seed()
├── data/mat/*.mat                   # 3064 extracted files
├── data/raw/cvind.mat, README.txt   # kept for provenance
├── manifest.csv                     # one row per image, with split column
├── leakage_check.json
└── run_info.json                    # library versions, counts, timestamp
```

## 1. Dependencies

`captum` (used for Integrated Gradients in notebook 2) is the only package that is actually
missing on current Kaggle images. `scikit-image`, `h5py` and `pyyaml` are usually already
installed, so pip will just confirm them. They stay in the list so the notebook still runs if
Kaggle changes its base image.

I'm not pinning versions here. I print them below and save them to `run_info.json` at the end,
so any result can be traced back to the environment it came from. That's the part that matters
for a write-up.

In [ ]:
!pip install -q captum scikit-image h5py pyyaml

In [ ]:
import json
import hashlib
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import sklearn
import skimage
import torch
import torchvision
import captum
import yaml
from skimage import measure
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

VERSIONS = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "captum": captum.__version__,
    "sklearn": sklearn.__version__,
    "skimage": skimage.__version__,
    "h5py": h5py.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
for k, v in VERSIONS.items():
    print(f"{k:12s} {v}")

## 2. GPU check

This notebook doesn't need a GPU, since it only does file I/O and plotting. I still check here
so that a missing accelerator shows up now and not 20 minutes into notebook 1.

If Kaggle gives you **T4 x2**, later notebooks use one GPU only. ResNet18 on ~2k training images
is small enough that `DataParallel` would add overhead and complicate the hooks Grad-CAM needs
(notebook 2). It wouldn't save meaningful time.

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i}  {torch.cuda.get_device_name(i)}")
    print("CUDA runtime:", torch.version.cuda, "| cuDNN:", torch.backends.cudnn.version())
else:
    DEVICE = torch.device("cpu")
    print("No GPU. Fine for this notebook; enable the accelerator before notebook 1.")

## 3. Project layout, config, and seeding

Everything lives under `/kaggle/working/` because that's the only writable directory whose
contents survive a **Save Version**.

All paths stored in the manifest are **relative to `DATA_ROOT`**, never absolute. When notebook 1
attaches this notebook's output as an input, it will be mounted read-only somewhere like
`/kaggle/input/<this-notebook-slug>/`, not `/kaggle/working/`. With relative paths, later
notebooks only have to change one `DATA_ROOT` line.

I also write the two small helper modules (`parse_mat.py`, `seed.py`) to `src/` with `%%writefile`.
They become part of this notebook's output, so later notebooks can import exactly the same code
and not paste copies that drift apart over time.

In [ ]:
WORK = Path("/kaggle/working")
DATA_ROOT = WORK / "data"
RAW_DIR = DATA_ROOT / "raw"   # zips while downloading, then just cvind.mat + README
MAT_DIR = DATA_ROOT / "mat"   # the 3064 .mat files, flat

for d in [RAW_DIR, MAT_DIR, WORK / "configs", WORK / "src" / "data", WORK / "src" / "utils", WORK / "results"]:
    d.mkdir(parents=True, exist_ok=True)
for pkg in [WORK / "src", WORK / "src" / "data", WORK / "src" / "utils"]:
    (pkg / "__init__.py").touch()

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

### 3a. Config

These are the values from `configs/baseline_resnet18.yaml`. **If your local YAML has fields I
haven't listed here (optimizer, weight decay, scheduler, num_workers, ...), paste your full
file over this cell.** The YAML is the source of truth; this cell just puts a copy where Kaggle
can see it.

Every downstream notebook loads its hyperparameters from this file and never hard-codes them,
so the baseline and the multi-task model in notebook 6 can't quietly end up with different
settings.

In [ ]:
%%writefile /kaggle/working/configs/baseline_resnet18.yaml
seed: 42
image_size: 224
batch_size: 32

data:
  manifest: manifest.csv        # relative to DATA_ROOT's parent (the notebook output dir)
  class_names: [meningioma, glioma, pituitary]   # index = raw label - 1
  split: {train: 0.70, val: 0.15, test: 0.15}

model:
  architecture: resnet18
  dropout_p: 0.3

train:
  epochs: 30
  lr: 0.0003

explain:
  methods: [gradcam, integrated_gradients]
  target_layer: layer4

audit:
  deletion_insertion_steps: 20
  stability_transforms:
    - {type: rotate, degrees: 5}
    - {type: rotate, degrees: -5}
    - {type: brightness, factor: 1.10}
  localization_threshold: 0.5

uncertainty:
  n_forward_passes: 30

In [ ]:
with open(WORK / "configs" / "baseline_resnet18.yaml") as f:
    CFG = yaml.safe_load(f)

SEED = CFG["seed"]
CLASS_NAMES = CFG["data"]["class_names"]
print(json.dumps(CFG, indent=2))

### 3b. Seeding

Nothing in this notebook is stochastic except the split and which images get plotted, and both
of those take `SEED` explicitly. The seed module is written here so notebooks 1–7 can share it.

`cudnn.deterministic=True` / `benchmark=False` costs a little training speed. I'll take that
trade: the audit in notebook 3 compares metric values between models, and I'd rather two runs
of the same model give the same numbers than save a few minutes per epoch.

In [ ]:
%%writefile /kaggle/working/src/utils/seed.py
import os
import random

import numpy as np
import torch


def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # only affects subprocesses (e.g. DataLoader workers); the current interpreter's hash seed is already fixed
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
from src.utils.seed import set_seed

set_seed(SEED)

## 4. Download the dataset from Figshare

Source: Cheng, J. *Brain tumor dataset*, Figshare, article 1512427 (CC BY 4.0). The article has
four zip parts (`brainTumorDataPublic_1-766.zip`, ..., `_2299-3064.zip`), `cvind.mat` (the
authors' 5-fold CV indices) and a `README.txt`.

Unlike `scripts/download_data.py`, which has the `ndownloader.figshare.com` URLs hard-coded, I
ask the Figshare API for the file list. The response also includes each file's size and MD5,
so I can verify each download against Figshare's own checksum. A truncated zip would otherwise
only show up as a confusing `BadZipFile` later.

If the API call fails (Figshare sometimes rate-limits cloud IPs), paste the URL dict from
`scripts/download_data.py` into `FALLBACK_URLS` and re-run; the MD5 check is skipped in that case.

The cell skips everything if the 3064 `.mat` files are already extracted, so it's safe to re-run.

In [ ]:
ARTICLE_ID = 1512427
N_EXPECTED = 3064
FALLBACK_URLS = {}   # {"filename": "https://ndownloader.figshare.com/files/<id>", ...} from download_data.py

# a plain requests UA occasionally gets a 403 from Figshare's CDN
HEADERS = {"User-Agent": "Mozilla/5.0 (research download; brain-tumor-xai-audit)"}


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, dest, size=None, md5=None):
    if dest.exists() and (md5 is None or md5sum(dest) == md5):
        print(f"have {dest.name}")
        return
    tmp = dest.with_name(dest.name + ".part")   # so an interrupted download never looks complete
    with requests.get(url, stream=True, timeout=60, headers=HEADERS) as r:
        r.raise_for_status()
        with open(tmp, "wb") as fh, tqdm(total=size, unit="B", unit_scale=True, desc=dest.name) as bar:
            for block in r.iter_content(chunk_size=1 << 20):
                fh.write(block)
                bar.update(len(block))
    if md5 is not None and md5sum(tmp) != md5:
        tmp.unlink()
        raise RuntimeError(f"MD5 mismatch for {dest.name}; re-run the cell")
    tmp.rename(dest)


if len(list(MAT_DIR.glob("*.mat"))) == N_EXPECTED:
    print(f"{N_EXPECTED} .mat files already extracted, skipping download")
else:
    if FALLBACK_URLS:
        files = [{"name": n, "download_url": u, "size": None, "computed_md5": None} for n, u in FALLBACK_URLS.items()]
    else:
        resp = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/files", timeout=30, headers=HEADERS)
        resp.raise_for_status()
        files = resp.json()

    for f in files:
        size_mb = f["size"] / 1e6 if f["size"] else float("nan")
        print(f"{f['name']:40s} {size_mb:8.1f} MB")
    for f in files:
        download(f["download_url"], RAW_DIR / f["name"], size=f["size"], md5=f["computed_md5"])

## 5. Extract

The four zips go into one flat `data/mat/` directory. Filenames are `1.mat` ... `3064.mat` across
all parts, so flattening is safe. I still check the final count below rather than assume it,
because a silent filename collision would cost images without any error.

The zips are deleted after extraction. They're the same data twice over, and everything in
`/kaggle/working/` counts toward the saved version's output size. `cvind.mat` and the README stay
for provenance; the README documents the `.mat` field layout that `parse_mat.py` depends on.

I don't use `cvind.mat` for splitting. The project plan calls for one held-out 70/15/15 split
(the audit and uncertainty notebooks need one fixed test set, not five folds), and I want the
patient grouping to be something I built and checked myself (section 7).

In [ ]:
for zp in sorted(RAW_DIR.glob("*.zip")):
    with zipfile.ZipFile(zp) as zf:
        members = [m for m in zf.infolist() if m.filename.endswith(".mat")]
        for m in tqdm(members, desc=zp.name, leave=False):
            # flatten whatever internal folder structure the zip has
            target = MAT_DIR / Path(m.filename).name
            if not target.exists():
                with zf.open(m) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    zp.unlink()

mat_files = sorted(MAT_DIR.glob("*.mat"), key=lambda p: int(p.stem))
print(len(mat_files), "mat files")
assert len(mat_files) == N_EXPECTED, f"expected {N_EXPECTED}, got {len(mat_files)}"
assert [int(p.stem) for p in mat_files] == list(range(1, N_EXPECTED + 1))

In [ ]:
# the README is short and documents the field layout -- worth reading once
readme = next(RAW_DIR.glob("README*"), None)
print(readme.read_text(errors="replace") if readme else "README not found")

## 6. Parsing one `.mat` file

The files are MATLAB v7.3, which is HDF5 underneath, so `scipy.io.loadmat` can't read them and
`h5py` can. Two things are less obvious than they look:

- **Transposes.** MATLAB stores arrays column-major; h5py hands them back with the axes reversed.
  For a 512×512 image you'd never notice from the shape, but the anatomy would be mirrored across
  the diagonal compared with how the images look in MATLAB/the paper. `.T` on both the image
  and the mask fixes it, and since both get the same treatment they stay aligned either way.
- **PID is a char array.** It arrives as a column of uint16 code points and has to be decoded
  back into a string. Some PIDs have trailing whitespace, hence the `.strip()`; without it, the
  same patient could end up counted as two "different" patients, which would break the
  leakage check.

`tumorBorder` is a flat `[x1, y1, x2, y2, ...]` vector according to the README. I reshape it to
`(K, 2)` and assume `(row, col)` pairs. The overlay in section 8 is what actually checks that.

In [ ]:
%%writefile /kaggle/working/src/data/parse_mat.py
from dataclasses import dataclass

import h5py
import numpy as np

LABEL_NAMES = {1: "meningioma", 2: "glioma", 3: "pituitary"}


@dataclass
class TumorSample:
    label: int                # raw 1/2/3
    patient_id: str
    image: np.ndarray         # (H, W) int16
    tumor_mask: np.ndarray    # (H, W) uint8 in {0, 1}
    tumor_border: np.ndarray  # (K, 2) polygon vertices, assumed (row, col)


def _decode_chars(ds) -> str:
    # MATLAB char arrays come through h5py as uint16 code points
    return "".join(chr(int(c)) for c in np.asarray(ds).ravel())


def read_mat(path) -> TumorSample:
    with h5py.File(path, "r") as f:
        cj = f["cjdata"]
        # h5py exposes MATLAB's column-major arrays with axes reversed
        image = np.asarray(cj["image"]).T
        mask = np.asarray(cj["tumorMask"]).T.astype(np.uint8)
        return TumorSample(
            label=int(np.asarray(cj["label"]).item()),
            patient_id=_decode_chars(cj["PID"]).strip(),
            image=image,
            tumor_mask=mask,
            tumor_border=np.asarray(cj["tumorBorder"]).ravel().reshape(-1, 2),
        )

In [ ]:
from src.data.parse_mat import read_mat, LABEL_NAMES, TumorSample

s = read_mat(mat_files[0])
print("label      ", s.label, LABEL_NAMES[s.label])
print("patient_id ", repr(s.patient_id))
print("image      ", s.image.shape, s.image.dtype, "range", s.image.min(), s.image.max())
print("tumor_mask ", s.tumor_mask.shape, np.unique(s.tumor_mask), "pixels:", int(s.tumor_mask.sum()))
print("border     ", s.tumor_border.shape)

## 7. Manifest and patient-level split

### Why split by patient

Each patient contributes several slices (glioma patients especially), and neighbouring slices
from one scan look almost the same. If images were split at random, the test set would hold
near-copies of training images, accuracy would be inflated, and, worse for this project, the
explanation audit would partly be measuring memorisation of specific patients' anatomy. So
**patients** are assigned to splits, and all of a patient's images follow them.

### How

1. Parse every file once, recording label, patient ID, tumor pixel count and image size.
2. Confirm every patient has exactly one label (verified before on your machine; re-checked here
   because this is a fresh download).
3. Build a one-row-per-patient table and split it **70 / 15 / 15**, stratified by label, with
   two calls to `train_test_split`: 70 vs 30, then the 30 split in half.

One caveat: stratification balances **patients** per class, not **images**. Patients contribute
different numbers of slices, so the image-level fractions will drift a little from 70/15/15.
I print both below so the drift is visible.

The parse loop reads every image in full, about a minute. That's deliberate: it also checks that
every file opens and that each mask's shape matches its image, before any training depends on it.

In [ ]:
rows = []
for p in tqdm(mat_files, desc="parsing"):
    s = read_mat(p)
    assert s.tumor_mask.shape == s.image.shape, p.name
    rows.append({
        "file_path": str(p.relative_to(DATA_ROOT)),
        "label": s.label,
        "label_name": LABEL_NAMES[s.label],
        "patient_id": s.patient_id,
        "tumor_pixel_count": int(s.tumor_mask.sum()),
        "image_h": s.image.shape[0],
        "image_w": s.image.shape[1],
    })

manifest = pd.DataFrame(rows)
print(manifest.shape)
print(manifest.label_name.value_counts().reindex(CLASS_NAMES))
print("\nimage sizes:")
print(manifest.groupby(["image_h", "image_w"]).size())
print("\nimages with an empty mask:", int((manifest.tumor_pixel_count == 0).sum()))

Expected: 708 / 1426 / 930 for meningioma / glioma / pituitary.

Most images are 512×512, but a small number are 256×256. That's why the manifest keeps `image_h`
/ `image_w`. The resize to 224 in the transforms makes this invisible to the model, but tumor
pixel counts aren't comparable across the two sizes without normalising by area (section 9 does that).

In [ ]:
labels_per_patient = manifest.groupby("patient_id")["label"].nunique()
assert (labels_per_patient == 1).all(), labels_per_patient[labels_per_patient > 1]

patients = (
    manifest.groupby("patient_id")
    .agg(label=("label", "first"), n_images=("file_path", "size"))
    .reset_index()
)
print(f"{len(patients)} patients (expected 233), every one with a single label")
print(patients.groupby("label").agg(patients=("patient_id", "size"), images=("n_images", "sum"),
                                    median_slices=("n_images", "median")))

In [ ]:
fr = CFG["data"]["split"]
train_p, rest_p = train_test_split(
    patients, test_size=fr["val"] + fr["test"], stratify=patients.label, random_state=SEED
)
val_p, test_p = train_test_split(
    rest_p, test_size=fr["test"] / (fr["val"] + fr["test"]), stratify=rest_p.label, random_state=SEED
)
# TODO: if the image-level test fraction lands far from 15%, try StratifiedGroupKFold-style
# assignment that balances images too -- the simple two-stage split was close enough last time

split_of = {pid: name for name, part in [("train", train_p), ("val", val_p), ("test", test_p)]
            for pid in part.patient_id}
manifest["split"] = manifest.patient_id.map(split_of)
assert manifest.split.notna().all()

### Leakage check

Three things have to hold:

- no patient ID appears in more than one split
- no file appears more than once in the manifest
- every patient is assigned somewhere

The result goes into `leakage_check.json`, so a later notebook (or a reviewer) can confirm the
split it's using passed, and doesn't have to take my word for it.

In [ ]:
pid_sets = {name: set(manifest.loc[manifest.split == name, "patient_id"]) for name in ["train", "val", "test"]}
overlaps = {
    "train&val": sorted(pid_sets["train"] & pid_sets["val"]),
    "train&test": sorted(pid_sets["train"] & pid_sets["test"]),
    "val&test": sorted(pid_sets["val"] & pid_sets["test"]),
}
leakage = {
    "patient_overlaps": overlaps,
    "duplicate_files": int(manifest.file_path.duplicated().sum()),
    "n_patients_per_split": {k: len(v) for k, v in pid_sets.items()},
    "all_patients_assigned": sum(len(v) for v in pid_sets.values()) == len(patients),
}
leakage["passed"] = (
    all(len(v) == 0 for v in overlaps.values())
    and leakage["duplicate_files"] == 0
    and leakage["all_patients_assigned"]
)
print(json.dumps(leakage, indent=2))
assert leakage["passed"], "patient leakage between splits"

with open(WORK / "leakage_check.json", "w") as f:
    json.dump(leakage, f, indent=2)

### Class balance per split

The first table shows image counts, the second shows each split's class proportions. The
proportions should be close across rows; if test had noticeably fewer meningiomas than train,
per-class audit metrics in notebook 3 would have wider error bars for that class. The last table
is the patient view, where stratification is exact up to rounding.

In [ ]:
order = ["train", "val", "test"]
counts = pd.crosstab(manifest.split, manifest.label_name).reindex(index=order, columns=CLASS_NAMES)
counts["total"] = counts.sum(axis=1)
counts["frac_of_images"] = (counts.total / counts.total.sum()).round(3)
display(counts)

display(pd.crosstab(manifest.split, manifest.label_name, normalize="index")
        .reindex(index=order, columns=CLASS_NAMES).round(3))

display(pd.crosstab(manifest.drop_duplicates("patient_id").split,
                    manifest.drop_duplicates("patient_id").label_name)
        .reindex(index=order, columns=CLASS_NAMES).rename_axis("patients"))

In [ ]:
cols = ["file_path", "label", "label_name", "patient_id", "tumor_pixel_count", "image_h", "image_w", "split"]
manifest[cols].to_csv(WORK / "manifest.csv", index=False)
print("wrote", WORK / "manifest.csv")
manifest[cols].head()

## 8. Visual sanity check

Four random images per class with:

- **red**: contour of `tumorMask` (via `skimage.measure.find_contours` at 0.5)
- **cyan dots**: the `tumorBorder` polygon vertices, under the `(row, col)` assumption

Things to look for:

1. The red contour sits on a visible lesion. This is the check that matters most: notebooks 3
   and 7 score explanation heatmaps against these masks, and a transposed or shifted mask would
   make every localization number meaningless while still producing plausible-looking values.
2. The cyan dots trace the red contour. If they trace a *mirrored* copy (reflected across the
   diagonal), the border pairs are `(col, row)`; swap the columns in `parse_mat.py`. Nothing
   downstream uses the border yet, so this only matters if it's used later.
3. Anatomy is the right way up and consistent across images. Pituitary tumors should sit near the
   midline at the skull base, and meningiomas usually against the skull or dura.

Images are shown with the same per-image min-max scaling `BrainTumorDataset` applies, so this
is the intensity range the model actually receives (before the resize to 224).

In [ ]:
def to_uint8(img):
    # same per-image min-max as BrainTumorDataset
    img = img.astype(np.float32)
    lo, hi = img.min(), img.max()
    return ((img - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)


N_PER_CLASS = 4
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(len(CLASS_NAMES), N_PER_CLASS, figsize=(3.3 * N_PER_CLASS, 3.5 * len(CLASS_NAMES)))
for row, cname in enumerate(CLASS_NAMES):
    sub = manifest[manifest.label_name == cname]
    picks = sub.iloc[rng.choice(len(sub), N_PER_CLASS, replace=False)]
    for ax, (_, r) in zip(axes[row], picks.iterrows()):
        s = read_mat(DATA_ROOT / r.file_path)
        ax.imshow(to_uint8(s.image), cmap="gray")
        for c in measure.find_contours(s.tumor_mask, 0.5):
            ax.plot(c[:, 1], c[:, 0], color="red", lw=1.2)
        ax.scatter(s.tumor_border[:, 1], s.tumor_border[:, 0], s=1.5, c="cyan")
        ax.set_title(f"{cname} | pid {r.patient_id}\n{Path(r.file_path).name} | {r.split} | {r.image_h}px", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

The same check on the hardest cases: the three smallest tumors per class, measured as a fraction
of image area. If a mask were off by a few pixels, these are the images where it would show,
and they are also where Grad-CAM's coarse 7×7 `layer4` resolution will struggle most at
localization (worth remembering when reading notebook 3's results).

In [ ]:
manifest["tumor_frac"] = manifest.tumor_pixel_count / (manifest.image_h * manifest.image_w)

fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(10, 3.5 * len(CLASS_NAMES)))
for row, cname in enumerate(CLASS_NAMES):
    smallest = manifest[manifest.label_name == cname].nsmallest(3, "tumor_frac")
    for ax, (_, r) in zip(axes[row], smallest.iterrows()):
        s = read_mat(DATA_ROOT / r.file_path)
        ax.imshow(to_uint8(s.image), cmap="gray")
        for c in measure.find_contours(s.tumor_mask, 0.5):
            ax.plot(c[:, 1], c[:, 0], color="red", lw=1.2)
        ax.set_title(f"{cname} | {Path(r.file_path).name}\ntumor = {100 * r.tumor_frac:.2f}% of image", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Tumor size distribution

This isn't strictly part of data setup, but it affects how notebook 3's localization metric
should be read. If pituitary tumors cover far less of the image than gliomas, then a heatmap
thresholded at `localization_threshold = 0.5` will look worse on pituitary even if the model
attends to the right place, because a blob from a 7×7 map is larger than the tumor. Seeing the
distribution now means that per-class differences later aren't misread as differences in model
behaviour.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(0, manifest.tumor_frac.quantile(0.99), 50)
for cname in CLASS_NAMES:
    ax.hist(manifest.loc[manifest.label_name == cname, "tumor_frac"], bins=bins, alpha=0.55, label=cname)
ax.set_xlabel("tumor area / image area")
ax.set_ylabel("images")
ax.legend()
plt.show()

manifest.groupby("label_name")["tumor_frac"].describe(percentiles=[0.05, 0.5, 0.95]).reindex(CLASS_NAMES).round(4)
# TODO: revisit localization_threshold once I see how Grad-CAM blobs compare to the 5th-percentile tumors

## 10. Run record and output check

In [ ]:
run_info = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "versions": VERSIONS,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seed": SEED,
    "figshare_article": ARTICLE_ID,
    "n_images": len(manifest),
    "n_patients": int(manifest.patient_id.nunique()),
    "images_per_split": manifest.split.value_counts().reindex(order).to_dict(),
    "patients_per_split": leakage["n_patients_per_split"],
    "leakage_check_passed": leakage["passed"],
}
with open(WORK / "run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)
print(json.dumps(run_info, indent=2))

In [ ]:
!du -sh /kaggle/working/data/mat /kaggle/working/data/raw
!ls -la /kaggle/working /kaggle/working/configs /kaggle/working/src/data /kaggle/working/src/utils

## Before you close this notebook: Save Version

`/kaggle/working/` is wiped when this session ends. To keep the manifest, the extracted data and
the `src/` modules for notebook 1:

1. Click **Save Version** (top right).
2. Choose **Save & Run All (Commit)** and click **Save**. Kaggle re-runs the whole notebook in a
   fresh session, from top to bottom. That also tests that the notebook really is
   self-contained and doesn't depend on some cell I ran by hand earlier.
3. Wait for the version to finish (the *Versions* panel shows the status), then open its
   **Output** tab and check that `manifest.csv`, `leakage_check.json`, `run_info.json`,
   `configs/`, `src/` and `data/mat/` (3064 files) are all there.

**In notebook 1:** use **Add Input → Your Work → Notebooks** and pick this notebook. Its output
is mounted read-only, at a path like `/kaggle/input/<this-notebook-slug>/` (run `!ls /kaggle/input`
to see the exact path). Notebook 1 points `DATA_ROOT` there, adds that directory to `sys.path` for
`src/`, and writes its own checkpoints to its own `/kaggle/working/`.

The commit run uses GPU quota if the accelerator is on. This notebook doesn't need one, so you
can set the accelerator to *None* before committing to save your weekly GPU hours.